In [6]:
import pandas as pd
inventory_raw=pd.read_csv('inventory.csv')
inventory = inventory_raw.copy()


In [25]:


# xóa khoảng trắng
inventory.columns = inventory.columns.str.strip()

# 3. Ép int
int_cols = [
    "product_id", "stock_on_hand", "units_received", "units_sold",
    "stockout_days", "stockout_flag", "overstock_flag", "reorder_flag",
    "year", "month"
]

for col in int_cols:
    inventory[col] = pd.to_numeric(
        inventory[col],
        errors="coerce"
    ).astype("Int64")

# 4. Ép float
float_cols = ["days_of_supply", "fill_rate", "sell_through_rate"]

for col in float_cols:
    inventory[col] = pd.to_numeric(
        inventory[col],
        errors="coerce"
    ).astype("float64")

# 5. Ép datetime
inventory["snapshot_date"] = pd.to_datetime(
    inventory["snapshot_date"],
    errors="coerce"
)

# 6. Xóa khoảng trắng thừa
string_cols = ["product_name", "category", "segment"]

for col in string_cols:
    inventory[col] = (
        inventory[col]
        .astype("string")
        .str.strip()
    )


inventory = inventory.dropna(subset=["snapshot_date", "product_id"])

inventory = inventory.drop_duplicates(
    subset=["product_id", "snapshot_date"]
).reset_index(drop=True)


column_order = [
    "snapshot_date", "product_id", "stock_on_hand", "units_received",
    "units_sold", "stockout_days", "days_of_supply", "fill_rate",
    "stockout_flag", "overstock_flag", "reorder_flag", "sell_through_rate",
    "product_name", "category", "segment", "year", "month"
]
inventory_cleaned = inventory[column_order]

# Check
print("--- inventory ---")
print(inventory_cleaned.info())
print("\n--- kiểm null ---")
print(inventory_cleaned.isnull().sum())
print("\nSố khóa trùng (product_id + snapshot_date):",
      inventory_cleaned.duplicated(subset=["product_id", "snapshot_date"]).sum())

inventory_cleaned.head()
inventory_cleaned.to_csv(
    "inventory_cleaned.csv",
    index=False,
    encoding="utf-8-sig"
)

--- inventory ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60247 entries, 0 to 60246
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   snapshot_date      60247 non-null  datetime64[ns]
 1   product_id         60247 non-null  Int64         
 2   stock_on_hand      60247 non-null  Int64         
 3   units_received     60247 non-null  Int64         
 4   units_sold         60247 non-null  Int64         
 5   stockout_days      60247 non-null  Int64         
 6   days_of_supply     60247 non-null  float64       
 7   fill_rate          60247 non-null  float64       
 8   stockout_flag      60247 non-null  Int64         
 9   overstock_flag     60247 non-null  Int64         
 10  reorder_flag       60247 non-null  Int64         
 11  sell_through_rate  60247 non-null  float64       
 12  product_name       60247 non-null  string        
 13  category           60247 non-null  string  

In [24]:
order_items_raw = pd.read_csv("order_items.csv")
order_items = order_items_raw.copy()
#  Cắt
order_items = order_items_raw[
    [
        "order_id",
        "product_id",
        "quantity",
        "unit_price",
        "discount_amount",
        "promo_id",
        "promo_id_2"
    ]
].copy()


text_cols = [
    "promo_id",
    "promo_id_2"
]

for col in text_cols:
    order_items[col] = (
        order_items[col]
        .fillna("NONE")
        .astype("string")
        .str.strip()
    )

int_cols = [
    "order_id",
    "product_id",
    "quantity"
]

for col in int_cols:
    order_items[col] = pd.to_numeric(
        order_items[col],
        errors="coerce"
    ).astype("Int64")

float_cols = [
    "unit_price",
    "discount_amount"
]

for col in float_cols:
    order_items[col] = pd.to_numeric(
        order_items[col],
        errors="coerce"
    )


order_items = order_items.dropna(
    subset=["order_id", "product_id"]
)

order_items = order_items.drop_duplicates(
    subset=["order_id", "product_id"]
).reset_index(drop=True)

order_items.head()

# 6. Check
print(order_items.isnull().sum())
print(
    "Số dòng trùng khóa chính (order_id + product_id):",
    order_items.duplicated(subset=["order_id", "product_id"]).sum()
)
order_items_cleaned = order_items.copy()
order_items_cleaned.to_csv(
    "order_items_cleaned.csv",
    index=False,
    encoding="utf-8-sig"
)

/tmp/ipykernel_1444/120423347.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items_raw = pd.read_csv("order_items.csv")


order_id           0
product_id         0
quantity           0
unit_price         0
discount_amount    0
promo_id           0
promo_id_2         0
dtype: int64
Số dòng trùng khóa chính (order_id + product_id): 0


In [22]:
payments_raw = pd.read_csv("payments.csv")
payments = payments_raw.copy()

# 2.xóa khoảng trắng thừa nếu có
payments.columns = payments.columns.str.strip()

# 3. Ép int
int_cols = [
    "order_id",
    "installments"
]

for col in int_cols:
    payments[col] = pd.to_numeric(
        payments[col],
        errors="coerce"
    ).astype("Int64")

# 4. Ép float
float_cols = [
    "payment_value"
]

for col in float_cols:
    payments[col] = pd.to_numeric(
        payments[col],
        errors="coerce"
    ).astype("float64")

# 5. Xóa khoảng trắng thừa
string_cols = [
    "payment_method"
]

for col in string_cols:
    payments[col] = (
        payments[col]
        .astype("string")
        .str.strip()
    )

# 6. Làm sạch
payments = payments.dropna(subset=["order_id"])

# Loại bỏ trùng lặp order_id (
payments = payments.drop_duplicates(
    subset=["order_id"]
).reset_index(drop=True)

# 7.
column_order = [
    "order_id",
    "payment_method",
    "payment_value",
    "installments"
]
payments_cleaned = payments[column_order]

# 8. Check

print(payments_cleaned.info())
print("\n--- kiểm null ---")
print(payments_cleaned.isnull().sum())
print("\nSố order_id trùng:", payments_cleaned.duplicated(subset=["order_id"]).sum())

payments_cleaned.head()
payments_cleaned.to_csv(
    "payments_cleaned.csv",
    index=False,
    encoding="utf-8-sig"
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   order_id        646945 non-null  Int64  
 1   payment_method  646945 non-null  string 
 2   payment_value   646945 non-null  float64
 3   installments    646945 non-null  Int64  
dtypes: Int64(2), float64(1), string(1)
memory usage: 21.0 MB
None

--- kiểm null ---
order_id          0
payment_method    0
payment_value     0
installments      0
dtype: int64

Số order_id trùng: 0


In [21]:

orders_raw = pd.read_csv("orders_enriched.csv")
orders = orders_raw.copy()

orders.columns = orders.columns.str.strip()

int_cols = [
    "order_id",
    "customer_id",
    "zip",
    "years_experience"
]

for col in int_cols:
    orders[col] = pd.to_numeric(
        orders[col],
        errors="coerce"
    ).astype("Int64")



orders["order_date"] = pd.to_datetime(
    orders["order_date"],
    errors="coerce"
)



string_cols = [
    "city",
    "region",
    "district",
    "order_status",
    "payment_method",
    "device_type",
    "order_source",
    "sales_employee_id",
    "sales_employee_name",
    "marital_status",
    "education_level",
    "comment"
]

for col in string_cols:
    orders[col] = (
        orders[col]
        .astype("string")
        .str.strip()
    )

category_cols = [
    "city",
    "region",
    "district",
    "order_status",
    "payment_method",
    "device_type",
    "order_source",
    "marital_status",
    "education_level"
]

for col in category_cols:
    orders[col] = (
        orders[col]
        .str.lower()
        .str.strip()
    )


orders = orders.replace(
    r"^\s*$",
    pd.NA,
    regex=True
)



orders = orders.dropna(
    subset=["order_id"]
)


# 9. Xử lý dữ liệu bất hợp lệ
# order_id và customer_id không được <= 0
# zip không được <= 0
# years_experience không được < 0
orders.loc[
    orders["order_id"] <= 0,
    "order_id"
] = pd.NA

orders.loc[
    orders["customer_id"] <= 0,
    "customer_id"
] = pd.NA

orders.loc[
    orders["zip"] <= 0,
    "zip"
] = pd.NA

orders.loc[
    orders["years_experience"] < 0,
    "years_experience"
] = pd.NA


# loại bỏ không còn order_id
orders = orders.dropna(
    subset=["order_id"]
)

# 10. Loại bỏ dữ liệu trùng lặp

orders = orders.drop_duplicates(
    subset=["order_id"]
).reset_index(drop=True)


column_order = [
    "order_id",
    "order_date",
    "customer_id",
    "zip",
    "city",
    "region",
    "district",
    "order_status",
    "payment_method",
    "device_type",
    "order_source",
    "sales_employee_id",
    "sales_employee_name",
    "marital_status",
    "education_level",
    "years_experience",
    "comment"
]

orders_cleaned = orders[column_order]

#Check


orders_cleaned.info()


print("\n--- kiểm null ---")

print(
    orders_cleaned.isnull().sum()
)



print(
    "Số order_id trùng:",
    orders_cleaned.duplicated(
        subset=["order_id"]
    ).sum()
)
orders_cleaned.to_csv(
    "orders_cleaned.csv",
    index=False,
    encoding="utf-8-sig"
)





<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 17 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             646945 non-null  Int64         
 1   order_date           646945 non-null  datetime64[ns]
 2   customer_id          646945 non-null  Int64         
 3   zip                  646945 non-null  Int64         
 4   city                 646945 non-null  string        
 5   region               646945 non-null  string        
 6   district             646945 non-null  string        
 7   order_status         646945 non-null  string        
 8   payment_method       646945 non-null  string        
 9   device_type          646945 non-null  string        
 10  order_source         646945 non-null  string        
 11  sales_employee_id    646945 non-null  string        
 12  sales_employee_name  646945 non-null  string        
 13  marital_status